<a href="https://colab.research.google.com/github/hamse-ai/Multimodal_Data_Preprocessing/blob/main/face_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Face Embeddings → CSV

This notebook takes a folder of people, 3 photos (neutral, smiling, and shocked) per person, detects each face, extracts a **face embedding vector** for it, and saves everything into a single CSV.


### What are we trying to do?

We use **DeepFace**, which handles two steps for us automatically:
1. **Face detection** which finds the faces inside each photo and crops or aligns them
2. **Embedding extraction** which runs the cropped face through a pretrained face-recognition model (we use `Facenet512`, which outputs a 512-number vector per face)

Two photos of the same person will produce embeddings that are close together in this 512-dimensional space; two different people will be far apart. That's what makes face matching/search possible later.

### **Install dependencies**


In [2]:
%pip install -q deepface pandas numpy tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.7/170.7 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.3/51.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 51.6 MB/s eta 0:00:00


###**Imports**

In [3]:
import os
import cv2
import glob
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
from deepface import DeepFace

26-07-19 14:58:27 - Directory /root/.deepface has been created
26-07-19 14:58:27 - Directory /root/.deepface/weights has been created


In [ ]:
import zipfile

zip_path = "/content/Pictures.zip"
extract_to = "/content/Pictures"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to)

print("Done! Files extracted to:", extract_to)

Done! Files extracted to: /content/Pictures


##**Configuration**

Change `DATASET_DIR` to point at your folder of people. Everything else has sensible defaults.

In [ ]:
# Path to the top-level folder that contains one subfolder per person
DATASET_DIR = "/content/dataset_augmented"

# Facenet512 = 512-dim vector, good accuracy/speed balance.
MODEL_NAME = "Facenet512"

# "opencv" is fastest face detector and has no extra dependencies.
DETECTOR_BACKEND = "opencv"

# Accepted image file extensions
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

# Where to save the resulting CSV
OUTPUT_CSV = "face_embeddings.csv"

##**Collect image paths per person**


In [ ]:
def collect_images(dataset_dir):
    """Return a dict: {person_name: [image_path, image_path, ...]}"""
    people = {}
    if not os.path.isdir(dataset_dir):
        raise FileNotFoundError(
            f"'{dataset_dir}' not found. Upload your dataset and/or update DATASET_DIR."
        )

    for person_name in sorted(os.listdir(dataset_dir)):
        person_dir = os.path.join(dataset_dir, person_name)
        if not os.path.isdir(person_dir):
            continue
        images = [
            os.path.join(person_dir, f)
            for f in sorted(os.listdir(person_dir))
            if f.lower().endswith(IMAGE_EXTENSIONS)
        ]
        if images:
            people[person_name] = images
    return people


people_images = collect_images(DATASET_DIR)

print(f"Found {len(people_images)} people:")
for name, imgs in people_images.items():
    print(f"  - {name}: {len(imgs)} image(s)")
    if len(imgs) < 2:
        print(f"    ⚠️  Warning: fewer than 2 images for '{name}'")

Found 4 people:
  - David faces: 12 image(s)
  - Hamse: 12 image(s)
  - Milka: 12 image(s)
  - Shoga: 12 image(s)


## **Augmentation**

In [ ]:
def rotate(img, max_angle=15):
    h, w = img.shape[:2]
    angle = random.uniform(-max_angle, max_angle)
    M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
    return cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REFLECT)

def horizontal_flip(img):
    return cv2.flip(img, 1)

def brightness_contrast(img, brightness_range=(-30, 30), contrast_range=(0.8, 1.2)):
    brightness = random.uniform(*brightness_range)
    contrast = random.uniform(*contrast_range)
    img = img.astype(np.float32)
    img = img * contrast + brightness
    return np.clip(img, 0, 255).astype(np.uint8)

def gaussian_noise(img, std=8):
    noise = np.random.normal(0, std, img.shape).astype(np.float32)
    noisy = img.astype(np.float32) + noise
    return np.clip(noisy, 0, 255).astype(np.uint8)

def gaussian_blur(img, max_kernel=3):
    k = random.choice([1, 3, max_kernel])  # 1 = no blur
    if k == 1:
        return img
    return cv2.GaussianBlur(img, (k, k), 0)

def random_zoom_crop(img, zoom_range=(0.85, 1.0)):
    h, w = img.shape[:2]
    scale = random.uniform(*zoom_range)
    new_h, new_w = int(h * scale), int(w * scale)
    top = random.randint(0, h - new_h)
    left = random.randint(0, w - new_w)
    cropped = img[top:top + new_h, left:left + new_w]
    return cv2.resize(cropped, (w, h))

AUGMENTATIONS = [rotate, horizontal_flip, brightness_contrast, gaussian_noise, gaussian_blur, random_zoom_crop]

def augment_image(img, num_augmentations=2):
    """Apply a random subset of augmentations to one image, return the result."""
    chosen = random.sample(AUGMENTATIONS, k=num_augmentations)
    for aug_fn in chosen:
        img = aug_fn(img)
    return img

def augment_dataset(dataset_dir, output_dir, copies_per_image=3, num_augmentations=2):
    """
    Walk dataset_dir (person subfolders), generate `copies_per_image` augmented
    versions of every image, and save them into output_dir with the same
    person subfolder structure. Originals are also copied over unchanged.
    """
    os.makedirs(output_dir, exist_ok=True)
    image_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

    for person_name in sorted(os.listdir(dataset_dir)):
        person_dir = os.path.join(dataset_dir, person_name)
        if not os.path.isdir(person_dir):
            continue

        out_person_dir = os.path.join(output_dir, person_name)
        os.makedirs(out_person_dir, exist_ok=True)

        for fname in sorted(os.listdir(person_dir)):
            if not fname.lower().endswith(image_extensions):
                continue

            img_path = os.path.join(person_dir, fname)
            img = cv2.imread(img_path)
            if img is None:
                print(f"Skipping unreadable file: {img_path}")
                continue

            base_name, ext = os.path.splitext(fname)

            # Copying the original unchanged
            cv2.imwrite(os.path.join(out_person_dir, fname), img)

            # Saving N augmented copies
            for i in range(copies_per_image):
                aug_img = augment_image(img.copy(), num_augmentations=num_augmentations)
                out_name = f"{base_name}_aug{i}{ext}"
                cv2.imwrite(os.path.join(out_person_dir, out_name), aug_img)

    print(f"Augmented dataset saved to: {output_dir}")


In [ ]:
augment_dataset("/content/Pictures/Pictures", "dataset_augmented", copies_per_image=3, num_augmentations=2)

Augmented dataset saved to: dataset_augmented


## **Extract embeddings**

For every image, `DeepFace.represent()`, detects the face(s) in the photo, crops and aligns each face, and runs it through the embedding model

If a photo has more than one face, we keep all of them since a group shot could otherwise mix people together. If no face is found, we skip the image and log it instead of crashing the whole run.

In [ ]:
def extract_embeddings(people_images, model_name=MODEL_NAME, detector_backend=DETECTOR_BACKEND):
    rows = []
    failures = []

    total_images = sum(len(v) for v in people_images.values())
    progress = tqdm(total=total_images, desc="Extracting embeddings")

    for person_name, image_paths in people_images.items():
        for image_path in image_paths:
            try:
                results = DeepFace.represent(
                    img_path=image_path,
                    model_name=model_name,
                    detector_backend=detector_backend,
                    enforce_detection=True,
                )
                # DeepFace.represent returns a list
                for face_index, face_result in enumerate(results):
                    rows.append({
                        "person": person_name,
                        "image_path": image_path,
                        "face_index": face_index,
                        "embedding": face_result["embedding"],
                    })
            except Exception as e:
                failures.append((image_path, str(e)))
            progress.update(1)

    progress.close()

    if failures:
        print(f"\n⚠️  {len(failures)} image(s) failed (usually: no face detected):")
        for path, err in failures:
            print(f"  - {path}: {err}")

    return rows


embedding_rows = extract_embeddings(people_images)
print(f"\nExtracted {len(embedding_rows)} face embeddings total.")

Extracting embeddings: 100%|██████████| 48/48 [05:34<00:00,  6.97s/it]


⚠️  4 image(s) failed (usually: no face detected):
  - /content/dataset_augmented/David faces/Smiling_aug1.jpg: Face could not be detected in /content/dataset_augmented/David faces/Smiling_aug1.jpg.Please confirm that the picture is a face photo or consider to set enforce_detection param to False.
  - /content/dataset_augmented/David faces/Smiling_aug2.jpg: Face could not be detected in /content/dataset_augmented/David faces/Smiling_aug2.jpg.Please confirm that the picture is a face photo or consider to set enforce_detection param to False.
  - /content/dataset_augmented/David faces/Surprised_aug1.jpg: Face could not be detected in /content/dataset_augmented/David faces/Surprised_aug1.jpg.Please confirm that the picture is a face photo or consider to set enforce_detection param to False.
  - /content/dataset_augmented/David faces/Surprised_aug2.jpg: Face could not be detected in /content/dataset_augmented/David faces/Surprised_aug2.jpg.Please confirm that the picture is a face photo o

## **Build the DataFrame and save to CSV**

Each embedding number gets its own column so the CSV stays flat and easy to load back with `pandas.read_csv`.

In [ ]:
def rows_to_dataframe(rows):
    records = []
    for row in rows:
        embedding = row["embedding"]
        record = {
            "person": row["person"],
            "image_path": row["image_path"],
            "face_index": row["face_index"],
        }
        for i, value in enumerate(embedding):
            record[f"emb_{i}"] = value
        records.append(record)
    return pd.DataFrame(records)


df = rows_to_dataframe(embedding_rows)
print(df.shape)
df.head()

(51, 515)


,person,image_path,face_index,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,emb_6,...,emb_502,emb_503,emb_504,emb_505,emb_506,emb_507,emb_508,emb_509,emb_510,emb_511
0,David faces,/content/dataset_augmented/David faces/Neutral...,0,0.648515,-0.867541,-1.216595,1.557090,0.345510,0.122260,0.493019,...,0.063310,-0.837503,-0.649396,1.413249,1.167381,0.084371,-0.302322,-0.805146,1.029646,-0.586055
1,David faces,/content/dataset_augmented/David faces/Neutral...,0,0.419869,-0.277404,-1.044728,1.505461,0.141757,0.482557,-0.049327,...,0.308646,-0.738844,-0.881808,1.000005,1.045212,-0.275716,-0.007649,-0.862313,1.476220,-0.275319
2,David faces,/content/dataset_augmented/David faces/Neutral...,0,0.499812,-0.763877,-1.100994,1.435944,0.383533,0.362983,0.741225,...,0.148107,-0.301068,-0.569230,0.919189,0.985637,-0.180162,-0.108353,-0.976203,0.659176,-0.729682
3,David faces,/content/dataset_augmented/David faces/Neutral...,0,0.427058,-0.846958,-1.158107,1.297353,0.382372,0.215961,0.705063,...,0.228896,-0.500478,-0.351911,0.844447,0.715317,-0.032887,-0.344315,-1.109924,0.858185,-0.307201
4,David faces,/content/dataset_augmented/David faces/Smiling...,0,0.419485,0.111346,-0.395287,1.033551,0.235924,0.730049,0.478543,...,1.891465,-0.175738,-1.544564,1.153734,0.520328,0.759236,-0.705889,-0.953653,0.179274,-0.601320


In [ ]:
df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(df)} rows to {OUTPUT_CSV}")

Saved 51 rows to face_embeddings.csv


## **Do same-person embeddings actually cluster together?**

This computes cosine similarity between every pair of faces. We should expect to see:
- **high similarity** (close to 1.0) between two photos of the *same* person
- **lower similarity** between different people


In [ ]:
def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


emb_cols = [c for c in df.columns if c.startswith("emb_")]

print(f"{'Image A':40s} {'Image B':40s} {'Same person?':13s} Similarity")
print("-" * 110)
for i in range(len(df)):
    for j in range(i + 1, len(df)):
        row_a, row_b = df.iloc[i], df.iloc[j]
        sim = cosine_similarity(row_a[emb_cols].values, row_b[emb_cols].values)
        same_person = row_a["person"] == row_b["person"]
        name_a = os.path.basename(row_a["image_path"])
        name_b = os.path.basename(row_b["image_path"])
        print(f"{name_a:40s} {name_b:40s} {str(same_person):13s} {sim:.3f}")

Image A                                  Image B                                  Same person?  Similarity
--------------------------------------------------------------------------------------------------------------
Neutral.jpg                              Neutral_aug0.jpg                         True          0.963
Neutral.jpg                              Neutral_aug1.jpg                         True          0.963
Neutral.jpg                              Neutral_aug2.jpg                         True          0.966
Neutral.jpg                              Smiling.jpg                              True          0.773
Neutral.jpg                              Smiling_aug0.jpg                         True          0.843
Neutral.jpg                              Surprised.jpg                            True          0.352
Neutral.jpg                              Surprised_aug0.jpg                       True          0.382
Neutral.jpg                              Hamse Neutral.jpg          